# Fig. 2: classification with a silent delay (MNIST, N-MNIST)

Every network has 100 hidden and 10 output neurons, feedforward, no recurrent connections. Trained with a 100 ms delay, then tested with delays up to 5 s. Nothing is retrained here: every number is read from

- `../MNIST/results/mnist_ms_experiments/` and
- `../N_MNIST/results/nmnist_w0_experiments/`

(`post_hoc/waiting/twait_sweep.csv`: accuracy against delay, 1280 test digits; `post_hoc/delay_activity/delay_activity.csv`: spikes per 100 ms, 512 test digits).

- **A.** MNIST test accuracy against delay. **B.** Hidden and output spikes for one MNIST digit, for the networks above 50 % at 5 s, with the rate per neuron between 4 and 5 s. **C.** As A, for N-MNIST.

Panel B runs one test digit through each network the first time (needs torch and the MNIST test set, downloaded by torchvision into `../MNIST/data/`), then reads the spikes from `cache/`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import style as S
S.apply()
import mnist_figures as M

data = M.load("MNIST")
data_nmnist = M.load("N-MNIST")

In [ ]:
fig, traces = M.make_figure_mnist_nmnist(data=data, data_nmnist=data_nmnist)
S.save(fig, "fig2_mnist_nmnist")
plt.show()

## Numbers in the text

Accuracy (%) after 0.1, 1 and 5 s of silence, and the rate (Hz per neuron, hidden + output) between 4 and 5 s.

In [ ]:
def delay_table(d):
    runs = M.runs_present(d)
    tab = pd.DataFrame({f"{t / 1000:g} s": M.accuracy_at(d["delay"], t) for t in (100, 1000, 5000)}).reindex(runs)
    tab["rate 4-5 s (Hz)"] = M.spike_rate(d["activity"]).reindex(runs)
    tab.index = [S.net(r)["label"] for r in runs]
    return tab.round(2)

display(delay_table(data))
display(delay_table(data_nmnist))
rate = M.spike_rate(data["activity"])
print(f"MNIST, rate 4-5 s: w0 = -1 {rate['const_w0_-1.00']:.2f} Hz, heterogeneous "
      f"{rate['lif_hidden_bistable_output_w0_-1.00']:.2f} Hz, ratio "
      f"{rate['const_w0_-1.00'] / rate['lif_hidden_bistable_output_w0_-1.00']:.0f}")

Learnt $w_0$ of the two-tailed network (hidden layer initialised half bistable, half IF-like; output layer around $-1$): share of bistable neurons before and after training, and how many neurons switched regime.

In [ ]:
display(M.w0_summary(dataset="MNIST"))
display(M.w0_summary(dataset="N-MNIST"))